# Spanish Transcritption

In [2]:
!pip install faster_whisper

In [1]:
!pip install pyannote.audio


In [3]:
!pip install noisereduce

## Imports


In [4]:
import numpy as np
import torch
import noisereduce as nr
import soundfile as sf
from pyannote.audio import Pipeline
from faster_whisper import WhisperModel
from pydub import AudioSegment
import tempfile, os

## Load and Resample

In [6]:
def load_and_resample(path: str, target_sr: int = 16000):
    audio = AudioSegment.from_file(path)
    audio = audio.set_channels(1).set_frame_rate(target_sr)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32)
    samples /= np.iinfo(audio.array_type).max   # normalize to [-1, 1]
    return samples, target_sr

## VAD

In [15]:
# Returns list of (start_sec, end_sec) tuples.
def run_vad(samples: np.ndarray, sr: int, hf_token: str):
    pipeline = Pipeline.from_pretrained(
        "pyannote/voice-activity-detection",
        token=hf_token
    )

    # Write temp wav so pyannote can read it
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        tmp_path = f.name
    sf.write(tmp_path, samples, sr)

    vad_result = pipeline(tmp_path)
    os.unlink(tmp_path)

    segments = [(seg.start, seg.end) for seg in vad_result.get_timeline()]
    print(f"[VAD] Found {len(segments)} speech segments")
    return segments

## Concatenate only the VAD speech chunks

In [8]:
def extract_speech_audio(samples: np.ndarray, sr: int, segments):
    chunks = [samples[int(s * sr): int(e * sr)] for s, e in segments]
    speech_audio = np.concatenate(chunks) if chunks else samples
    print(f"[VAD] Kept {len(speech_audio)/sr:.1f}s of speech "
          f"from {len(samples)/sr:.1f}s total")
    return speech_audio

## Noise Reduction

In [9]:
def reduce_noise(samples: np.ndarray, sr: int):
    """Spectral noise reduction (noisereduce)."""
    denoised = nr.reduce_noise(y=samples, sr=sr, stationary=False, prop_decrease=0.8)
    print("[Noise Reduction] Done")
    return denoised

## Transcribe

In [10]:
def transcribe(samples: np.ndarray, sr: int, language=None):
    """Transcribe with faster-whisper medium, auto or forced language."""
    model = WhisperModel("medium", device="cuda", compute_type="float16")

    segments, info = model.transcribe(
        samples,                    # accepts numpy array directly
        language=language,          # None = auto-detect per segment
        beam_size=5,
        vad_filter=False,           # already did VAD manually
        word_timestamps=True,
    )

    print(f"[Whisper] Detected language: {info.language} "
          f"(p={info.language_probability:.2f})")

    results = []
    for seg in segments:
        results.append({
            "start": seg.start,
            "end":   seg.end,
            "text":  seg.text.strip(),
            "lang":  info.language,
        })
        print(f"  [{seg.start:.1f}s → {seg.end:.1f}s] {seg.text.strip()}")

    return results

## Keeping only Spanish

In [11]:
def filter_spanish(results):
    """Keep only segments detected as Spanish (useful if language=None)."""
    spanish = [r for r in results if r["lang"] == "es"]
    print(f"[Filter] {len(spanish)}/{len(results)} segments are Spanish")
    return spanish

## Pipeline

In [17]:
# ─── CONFIG ───────────────────────────────────────────────────────────────────
AUDIO_PATH     = "/content/drive/MyDrive/spanish/038010_EIT-2A.mp3"   # any format
TARGET_SR      = 16000                    # Whisper expects 16kHz
HF_TOKEN       = ""         # for pyannote VAD
TRANSCRIBE_LANG = None                    # None = auto-detect; "es" = force Spanish
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # 1. Load + resample
    audio, sr = load_and_resample(AUDIO_PATH, TARGET_SR)

    # 2. VAD — strip silence/noise chunks
    speech_segments = run_vad(audio, sr, HF_TOKEN)
    audio = extract_speech_audio(audio, sr, speech_segments)

    # 3. Noise reduction on speech-only audio
    audio = reduce_noise(audio, sr)

    # 4. Transcribe (auto-detect language per segment)
    results = transcribe(audio, sr, language=TRANSCRIBE_LANG)

    # 5. Post-process: keep Spanish only (or skip if you forced language="es")
    spanish_results = filter_spanish(results)

    # 6. Final transcript
    full_transcript = " ".join(r["text"] for r in spanish_results)
    print("\n─── SPANISH TRANSCRIPT ───")
    print(full_transcript)

config.yaml:   0%|          | 0.00/277 [00:00<?, ?B/s]

ValueError: Revisions must be passed with `revision` keyword argument.